# Task 2.1 — Dataset Selection and Setup
**Paper:** Poisoning Attacks against Support Vector Machines  
**Authors:** Battista Biggio, Blaine Nelson, Pavel Laskov (ICML 2012)  
**Roll Number:** 230110  
**Random Seed:** 42

---
## Dataset Justification

We use the **Breast Cancer Wisconsin (Diagnostic)** dataset from scikit-learn (`sklearn.datasets.load_breast_cancer`).
This dataset contains 569 samples with 30 real-valued features computed from digitised images of breast mass
fine-needle aspirate (FNA) biopsies, and the task is binary classification (malignant vs benign).

**Why it is appropriate for this paper's method:**  
The paper proposes a gradient-based poisoning attack against SVMs on binary classification problems.
The Breast Cancer dataset is a well-separated binary classification task on which SVMs achieve high accuracy,
making it a suitable testbed to observe the degradation caused by injecting poisoned training points.
The 30-dimensional feature space is rich enough for the gradient-based attack to operate meaningfully,
while being small enough for CPU-only computation without any GPU requirements.

**Limitations compared to the original paper's dataset:**  
The paper evaluates on MNIST digit pairs (e.g., 7 vs 1) with 784 features and image-space structure.
Our dataset has only 30 features and lacks the spatial/visual semantics of images, so the "visual morphing"
effect shown in the paper's Figure 2 (where attack points visually resemble the opposing class) cannot be
observed. Additionally, MNIST subsets have 100 training and 500 validation samples per experiment; our
smaller dataset (100 train, 100 val) provides less gradient signal.

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [2]:
# Load dataset
data = load_breast_cancer()
X, y = data.data, data.target

print(f"Dataset: Breast Cancer Wisconsin (Diagnostic)")
print(f"Total samples: {X.shape[0]}")
print(f"Features: {X.shape[1]}")
print(f"Classes: {np.unique(y)} — 0=malignant ({np.sum(y==0)}), 1=benign ({np.sum(y==1)})")

Dataset: Breast Cancer Wisconsin (Diagnostic)
Total samples: 569
Features: 30
Classes: [0 1] — 0=malignant (212), 1=benign (357)


The code above loads the Breast Cancer dataset and displays its basic statistics — 569 samples, 30 features,
and 2 classes (357 benign, 212 malignant). This matches the requirement of at least 100 samples and 2 features.

In [3]:
# Preprocessing: normalise features to [0, 1] and convert labels to +1/-1
# (SVM convention: +1 for benign, -1 for malignant)
y = 2 * y - 1  # 0 -> -1 (malignant), 1 -> +1 (benign)

scaler = MinMaxScaler()
X = scaler.fit_transform(X)

print(f"Feature range after scaling: [{X.min():.1f}, {X.max():.1f}]")
print(f"Label distribution: +1 (benign)={np.sum(y==1)}, -1 (malignant)={np.sum(y==-1)}")

Feature range after scaling: [0.0, 1.0]
Label distribution: +1 (benign)=357, -1 (malignant)=212


Features are normalised to $[0, 1]$ using MinMaxScaler. This is consistent with the paper's preprocessing
of MNIST data, where pixel values are divided by 255 to obtain the same range (Section 3.2).
Labels are converted to $\{+1, -1\}$ to match the SVM formulation used in the paper.

In [4]:
# Split into train (100), validation (100), and test (remaining ~369) sets
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.35, random_state=RANDOM_SEED, stratify=y
)
X_train, X_rem, y_train, y_rem = train_test_split(
    X_temp, y_temp, train_size=100, random_state=RANDOM_SEED
)
X_val = X_rem[:100]
y_val = y_rem[:100]

print(f"\nData split:")
print(f"  Training:   {X_train.shape[0]} samples (malignant={np.sum(y_train==-1)}, benign={np.sum(y_train==1)})")
print(f"  Validation: {X_val.shape[0]} samples (malignant={np.sum(y_val==-1)}, benign={np.sum(y_val==1)})")
print(f"  Testing:    {X_test.shape[0]} samples (malignant={np.sum(y_test==-1)}, benign={np.sum(y_test==1)})")


Data split:
  Training:   100 samples (malignant=37, benign=63)
  Validation: 100 samples (malignant=43, benign=57)
  Testing:    200 samples (malignant=75, benign=125)


The data is split into 100 training, 100 validation, and ~200 test samples.
This split mirrors the paper's experimental setup (Section 3.2) where 100 training and 500 validation
samples are used for each MNIST digit pair. Our smaller total dataset (569 samples) limits the
validation and test set sizes, but the training size matches the paper exactly.

In [5]:
# Visualisation: feature distribution for first two features
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for i, (ax, title) in enumerate(zip(axes, ['Feature 0: Mean Radius', 'Feature 1: Mean Texture'])):
    for label, color, name in [(-1, '#F44336', 'Malignant'), (1, '#2196F3', 'Benign')]:
        mask = y_train == label
        ax.hist(X_train[mask, i], bins=15, alpha=0.6, color=color, label=name, edgecolor='white')
    ax.set_xlabel(title, fontsize=11)
    ax.set_ylabel('Count', fontsize=11)
    ax.legend()
    ax.grid(True, alpha=0.3)

fig.suptitle('Feature Distributions in Training Set', fontsize=13)
plt.tight_layout()
plt.savefig('/Users/yashlunawat/C/sem6/AML/230110-midsem/partB/results/dataset_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to partB/results/dataset_feature_distributions.png")

Saved to partB/results/dataset_feature_distributions.png


The histograms show the distribution of the first two features (mean radius and mean texture) across
the two classes in the training set. The classes are reasonably separable, making this a good candidate
for SVM classification — and hence a meaningful target for poisoning attacks.